<div style="
background: linear-gradient(135deg, #f8f9fa 0%, #edf6f9 45%, #e8eaf6 100%);
padding: 40px;
border-radius: 20px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 8px 24px rgba(0,0,0,0.08);
border: 1px solid #dce3ea;
">

  <h1 style="
  color: #5c6b8a;
  font-size: 2.2em;
  margin: 0 0 8px 0;
  letter-spacing: 1px;
  font-weight: 700;">
  🤖 CP020003 — Artificial Intelligence 2026
  </h1>

  <h2 style="
  color: #7b8fa1;
  font-size: 1.3em;
  margin: 0 0 16px 0;
  font-weight: 400;">
  Khon Kaen University
  </h2>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    👨‍🏫 <strong style="color:#6c7aa1;">Author:</strong>
    Teerapong Panboonyuen (P'Kao)
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📧 <strong style="color:#6c7aa1;">Contact:</strong>
    teerapong.pa@chula.ac.th
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    🏫 <strong style="color:#6c7aa1;">Course:</strong>
    AI 2026 @ KKU
  </p>

  <p style="color: #495057; font-size: 1.05em; margin: 6px 0;">
    📦 <strong style="color:#6c7aa1;">GitHub:</strong>
    <a href="https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
       style="color:#5b8def; text-decoration:none;">
       CP020003_ArtificialIntelligence_2026s1
    </a>
  </p>

  <hr style="
  border: 1px solid #c9d6df;
  width: 60%;
  margin: 18px auto;">

  <p style="
color: #6c757d;
font-size: 0.95em;
margin: 4px 0;">
📚 Built with inspiration from the open-source AI community:
<strong style="color:#7286a0;">
Python · Pandas · NumPy · scikit-learn · PyTorch · Hugging Face · Kaggle
</strong>
</p>

  <p style="
  color: #8a97a6;
  font-size: 0.9em;
  margin-top: 12px;
  font-style: italic;">
  "This notebook is open to everyone — including those who cannot afford university.
  Knowledge is for all. 🌏"
  </p>

</div>

**Dataset:** [Spotify KKU Dataset](https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/blob/main/dataset/spotify-kku-dataset.csv)

> 🧠 **Big idea of this week:** *"Garbage in, garbage out."* No matter how fancy your model is next week (Week 3 — Supervised Learning), if your **features** are weak, your model will be weak. This week is 100% about **creating great features** — the raw material of every winning Kaggle/Hackathon solution.
>
> 🚀 This notebook is written to run **comfortably on CPU** in Google Colab (Runtime → Change runtime type → **CPU** is enough — no GPU required!). All HuggingFace models used here are small/distilled models chosen specifically for CPU speed.

---

## 📑 Table of Contents

0. [Setup & Installs](#0)
1. [Load & Inspect the Dataset](#1)
2. [🔁 Quick Recap: Python Functions, Lambda & Logic (the tools we use to build features)](#2)
3. [🔢 Basic Numeric Feature Engineering](#3)
4. [➗ Ratio & Interaction Features](#4)
5. [🗂️ Binning & Categorical Encoding](#5)
6. [📝 Text-Based Features (artists, track/album names)](#6)
7. [📊 Aggregation Features (Group Statistics / Target-style Encoding)](#7)
8. [🎼 Music-Theory-Inspired Features (mood quadrant, musical key, mode)](#8)
9. [⚖️ Feature Scaling & Transformation (preview for next week)](#9)
10. [🤗 HuggingFace Zero-Shot Mood Classification (CPU-friendly)](#10)
11. [🤗 HuggingFace Sentence Embeddings for Track Names (CPU-friendly)](#11)
12. [📉 Dimensionality Reduction of Embeddings (PCA) as New Features](#12)
13. [🧩 Putting It All Together: The Final Feature Table](#13)
14. [💾 Save the Engineered Dataset](#14)
15. [✅ Recap & What's Next (Week 4: Supervised Learning)](#15)
16. [📝 This Week's Assignment — 10 Feature Engineering Challenges](#16)

---


<a id="0"></a>
## 0. Setup & Installs

We install:
- `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn` → classic data science stack
- `transformers`, `sentence-transformers` → to bring in **free, open-source LLM-era tools** from HuggingFace 🤗

💡 All models chosen in this notebook are **small/distilled**, so they run fine on CPU. If you have a GPU (e.g. Colab Pro), everything will simply run faster — no code changes needed.


In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_style("whitegrid")
print("✅ Imports ready. pandas version:", pd.__version__)

In [ ]:
# Import HuggingFace's high-level pipeline API
from transformers import pipeline

<a id="1"></a>
## 1. Load & Inspect the Dataset

We load the CSV directly from GitHub (raw link), so this notebook works out-of-the-box on Colab with zero manual upload.


In [ ]:
YOUR_DATASET_NAME = 'INSERT-YOUR-DATASET-NAME'
RAW_URL = f"https://raw.githubusercontent.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/main/dataset/{YOUR_DATASET_NAME}.csv"

df = pd.read_csv(RAW_URL, index_col=0)
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Number of unique genres:", df["track_genre"].nunique())
df["track_genre"].value_counts().head(10)

<a id="2"></a>
## 2. 🔁 Quick Recap: Python Functions, Lambda & Logic

Feature engineering is really just **writing small functions that transform columns into new columns**. Let's recap the exact tools we will use over and over in this notebook.

### 2.1 `def` — a normal Python function
A regular function is reusable, testable, and can hold complex logic (if/elif/else, loops, etc).


In [ ]:
def ms_to_minutes(duration_ms):
    # Convert milliseconds to minutes (float)
    return # WRITE YOUR CODE HERE

# quick test
print(ms_to_minutes(230666))

### 2.2 `lambda` — a tiny anonymous function
Great for short, one-line transformations, especially inside `.apply()`.


In [ ]:
df["duration_min_demo"] = # WRITE YOUR CODE HERE
df[["duration_ms", "duration_min_demo"]].head()

### 2.3 `if / elif / else` inside a function — the heart of "rule-based" features
Most "smart" categorical features in a hackathon are literally just if-else logic wrapped in a function.


In [ ]:
def tempo_category(bpm):
    # Classify a song's tempo (beats per minute) into a human category
    return # WRITE YOUR CODE HERE

# test on a few values
for t in [70, 100, 150]:
    print(t, "->", tempo_category(t))

### 2.4 `.map()` vs `.apply()` vs `.assign()` — three ways to create a column

| Method | Best for | Example |
|---|---|---|
| `.map()` | Element-wise mapping, Series only, dict or function | `df["mode"].map({0: "minor", 1: "major"})` |
| `.apply()` | Apply a function row-wise or element-wise, Series or DataFrame | `df["tempo"].apply(tempo_category)` |
| `.assign()` | Chain-create multiple new columns cleanly (great for pipelines) | `df.assign(new_col=lambda d: d["x"] * 2)` |

### 2.5 List comprehension — compact way to loop
`[expression for item in iterable if condition]`


In [ ]:
# Example: list comprehension to quickly check which tracks are "very danceable" (danceability > 0.8)
very_danceable_flags = [1 if d > 0.8 else 0 for d in df["danceability"]]
print("Number of very danceable tracks:", sum(very_danceable_flags))

# cleanup demo column
df.drop(columns=["duration_min_demo"], inplace=True)

👉 **Rule of thumb:** simple math transform → `lambda` inside `.apply()` or `.assign()`. Multi-branch business logic → `def` function with `if/elif/else`, then `.apply()` or `.map()`.

Now let's use these tools to build **real features**!


<a id="3"></a>
## 3. 🔢 Basic Numeric Feature Engineering

Simple unit conversions and rescalings that make raw numbers more model-friendly and human-readable.


In [ ]:
# Convert duration from milliseconds -> minutes (much easier for a human to read)
df["duration_min"] = # WRITE YOUR CODE HERE

In [ ]:
# Loudness is stored as negative dB (e.g. -6.7). Taking abs() makes "louder = bigger number"
df["loudness_abs"] = # WRITE YOUR CODE HERE

In [ ]:
# Rescale popularity from a 0-100 score down to a 0-1 range
df["popularity_norm"] = # WRITE YOUR CODE HERE

In [ ]:
# Round tempo (BPM) to the nearest whole number, just to make it tidier to read
df["tempo_rounded"] = # WRITE YOUR CODE HERE

In [ ]:
# 👀 Let's check all 4 new columns side-by-side with their originals
df[["duration_ms", "duration_min", "loudness", "loudness_abs",
    "popularity", "popularity_norm", "tempo", "tempo_rounded"]].head()

<a id="4"></a>
## 4. ➗ Ratio & Interaction Features

These are the classic **"hackathon gold"** features — combining two or more columns often reveals patterns a single column can't.


In [ ]:
# How much energy per unit of loudness? (+1e-6 just avoids dividing by zero)
df["energy_loudness_ratio"] = # WRITE YOUR CODE HERE

In [ ]:
# A combined "party score": high only when BOTH danceability AND energy are high
df["dance_energy_score"] = # WRITE YOUR CODE HERE

In [ ]:
# Popularity earned per minute of song length (short catchy songs vs long popular songs)
df["popularity_per_minute"] = # WRITE YOUR CODE HERE

In [ ]:
# Average of two "quiet/organic" traits into one single index
df["acoustic_instr_index"] = # WRITE YOUR CODE HERE

In [ ]:
# Positive = lots of spoken words vs instrumental; negative = mostly instrumental
df["vocal_energy_gap"] = # WRITE YOUR CODE HERE

In [ ]:
# 👀 Check all 5 new interaction/ratio features together
df[["energy_loudness_ratio", "dance_energy_score", "popularity_per_minute",
    "acoustic_instr_index", "vocal_energy_gap"]].head()

<a id="5"></a>
## 5. 🗂️ Binning & Categorical Encoding

Turning continuous numbers into buckets often makes patterns easier for tree-based models (and humans!) to find. We also decode Spotify's numeric `key` and `mode` into real musical terms.


In [ ]:
# Duration bucket: cut minutes into 3 human-readable groups
df["duration_bucket"] = pd.cut(
    df["duration_min"],
    bins=[0, 2.5, 4, 100],
    labels=["short", "medium", "long"]
)

In [ ]:
# Tempo bucket: reuse our tempo_category() function from Section 2!
df["tempo_bucket"] = # WRITE YOUR CODE HERE

In [ ]:
# Musical key decoding: Spotify encodes key as a number, 0=C, 1=C#/Db, ... 11=B
key_map = # WRITE YOUR CODE HERE

In [ ]:
# Apply the key_map dictionary using .map() -> turns 0,1,2... into real note names
df["key_name"] = # WRITE YOUR CODE HERE

In [ ]:
# Mode decoding: Spotify uses 0 = minor scale, 1 = major scale
df["mode_name"] = # WRITE YOUR CODE HERE

In [ ]:
# 👀 Check all the new binned/decoded columns together
df[["duration_min", "duration_bucket", "tempo", "tempo_bucket",
    "key", "key_name", "mode", "mode_name"]].head(10)

In [ ]:
# quick visual sanity check
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["duration_bucket"].value_counts().plot(kind="bar", ax=axes[0], title="Duration Bucket")
df["tempo_bucket"].value_counts().plot(kind="bar", ax=axes[1], title="Tempo Bucket")
plt.tight_layout()
plt.show()

<a id="6"></a>
## 6. 📝 Text-Based Features (artists, track & album names)

Even without any AI model, plain **string logic** on `artists` and `track_name` gives surprisingly strong features — this is a favorite trick in music/recommendation hackathons.


In [ ]:
def count_artists(artist_string):
    # Spotify separates collaborating artists with ';' -> count them
    return len(str(artist_string).split(";"))

In [ ]:
# Number of artists credited on the track (using our count_artists() function above)
df["num_artists"] = # WRITE YOUR CODE HERE

In [ ]:
# Flag: is this a collaboration (2+ artists) or a solo track?
df["is_collab"] = # WRITE YOUR CODE HERE

In [ ]:
# How many characters are in the track title?
df["track_name_length"] = # WRITE YOUR CODE HERE

In [ ]:
# How many words are in the track title?
df["track_word_count"] = # WRITE YOUR CODE HERE

In [ ]:
# Flag: does the title mention "acoustic"? (e.g. "Ghost - Acoustic")
df["is_acoustic_version"] = df["track_name"].astype(str).str.contains(r"acoustic", case=False, regex=True).astype(int)

In [ ]:
# Flag: does the title mention "remix" / "mix" / "edit"?
df["is_remix"] = df["track_name"].astype(str).str.contains(r"remix|mix|edit", case=False, regex=True).astype(int)

In [ ]:
# Flag: does the title mention "live" (e.g. recorded live)?
df["is_live"] = df["track_name"].astype(str).str.contains(r"live", case=False, regex=True).astype(int)

In [ ]:
# Flag: does the title contain parentheses or brackets, e.g. "(feat. ...)"
df["has_parenthesis"] = df["track_name"].astype(str).str.contains(r"[\(\[]", regex=True).astype(int)

In [ ]:
# 👀 Check all the new text-based features together
df[["artists", "num_artists", "is_collab", "track_name", "track_name_length",
    "track_word_count", "is_acoustic_version", "is_remix", "is_live", "has_parenthesis"]].head(10)

👉 Notice: **`is_acoustic_version`** built from *text* often correlates very strongly with the numeric **`acousticness`** column. That's a great discussion point for students: *"Can a cheap text feature approximate an expensive audio-analysis feature?"*


<a id="7"></a>
## 7. 📊 Aggregation Features (Group Statistics)

"Target-style" and group-based features are extremely powerful in tabular ML competitions: *"How does this row compare to others in the same group?"*

⚠️ **Teaching note:** using the target column (`popularity`) to build group features can cause **data leakage** if not done carefully (e.g. with cross-validation folds). We build them here for teaching purposes — next week we'll discuss safe ways to do this in a real modeling pipeline.


In [ ]:
# Average popularity per genre, broadcast back to every row in that genre
df["genre_avg_popularity"] = # WRITE YOUR CODE HERE

In [ ]:
# Rank of danceability within each genre (1 = most danceable in that genre)
df["dance_rank_in_genre"] = df.groupby("track_genre")["danceability"].rank(ascending=False)

In [ ]:
# Get each genre's average popularity (needed for the z-score in the next cell)
genre_mean = df.groupby("track_genre")["popularity"].transform("mean")

In [ ]:
# Get each genre's popularity standard deviation (spread) -- also needed for the z-score
genre_std = df.groupby("track_genre")["popularity"].transform("std")

In [ ]:
# How popular is this song vs the average of its own genre? (z-score style)
df["popularity_z_in_genre"] = (df["popularity"] - genre_mean) / (genre_std + 1e-6)

In [ ]:
# Artist-level average popularity (a simple, illustrative "target encoding")
artist_pop = df.groupby("artists")["popularity"].transform("mean")

In [ ]:
# Attach the artist-average-popularity we just computed as a new column
df["artist_avg_popularity"] = artist_pop

In [ ]:
# 👀 Check all the new aggregation/group features together
df[["track_genre", "popularity", "genre_avg_popularity", "dance_rank_in_genre",
    "popularity_z_in_genre", "artists", "artist_avg_popularity"]].head(10)

<a id="8"></a>
## 8. 🎼 Music-Theory-Inspired Features: Mood Quadrant

This is inspired by **Russell's Circumplex Model of Affect**, commonly used in music-emotion research: plotting songs on a 2D plane of **Valence** (musical positivity) vs **Energy**.

| | High Energy | Low Energy |
|---|---|---|
| **High Valence** | 😄 Happy / Excited | 😌 Calm / Peaceful |
| **Low Valence** | 😡 Angry / Tense | 😢 Sad / Depressed |


In [ ]:
def mood_quadrant(valence, energy):
    # Classify a song's mood using the valence-energy circumplex model
    return # WRITE YOUR CODE HERE

In [ ]:
# Apply mood_quadrant() row-by-row (axis=1 means "one full row at a time")
df["mood_quadrant"] = df.apply(lambda row: mood_quadrant(row["valence"], row["energy"]), axis=1)

In [ ]:
# 👀 Check the new mood_quadrant column
df[["track_name", "valence", "energy", "mood_quadrant"]].head(10)

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(data=df.sample(1500, random_state=42), x="valence", y="energy",
                 hue="mood_quadrant", alpha=0.6, s=20)
plt.axvline(0.5, color="gray", linestyle="--")
plt.axhline(0.5, color="gray", linestyle="--")
plt.title("Mood Quadrant: Valence vs Energy")
plt.show()

<a id="9"></a>
## 9. 🤗 HuggingFace Zero-Shot Mood Classification (CPU-friendly)

Welcome to the **LLM era of feature engineering**! Instead of hand-writing rules, we can ask a pretrained language model to *label* text for us — with **zero training examples** ("zero-shot").

We use `typeform/distilbert-base-uncased-mnli` — a **small, distilled** model (~250MB) that runs fast on CPU. (The famous `facebook/bart-large-mnli` also works, but it's ~1.6GB and much slower on CPU — great to try at home with more compute/GPU.)

We'll demo this on a small sample of songs (zero-shot inference is relatively slow row-by-row) — feel free to scale up on GPU for your hackathon project.


In [ ]:
# Import HuggingFace's high-level pipeline API
from transformers import pipeline

In [ ]:
# Load a small, CPU-friendly zero-shot classification model (~250MB)
zero_shot = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",
    device=-1,  # -1 = force CPU
)

In [ ]:
# Define our own candidate mood labels -- the model was never trained on these specifically!
candidate_moods = ["happy", "sad", "romantic", "energetic", "chill", "aggressive"]

In [ ]:
# Take a small random sample of 15 songs (zero-shot inference is slower row-by-row)
sample_df = df.sample(15, random_state=42).copy()

In [ ]:
def classify_mood_llm(track_name, artist_name):
    # Ask the zero-shot model to pick the best-matching mood label for this song's title
    text = f"{track_name} by {artist_name}"
    result = zero_shot(text, candidate_labels=candidate_moods)
    return result["labels"][0], result["scores"][0]

In [ ]:
# Apply our classify_mood_llm() function to every row in the sample
results = sample_df.apply(
    lambda row: classify_mood_llm(row["track_name"], row["artists"]), axis=1
)

In [ ]:
# Split the (label, score) tuples into two separate columns
sample_df["llm_mood_label"] = results.apply(lambda r: r[0])
sample_df["llm_mood_score"] = results.apply(lambda r: r[1])

In [ ]:
# 👀 Compare our simple audio-based mood_quadrant vs the LLM's zero-shot mood guess
sample_df[["track_name", "artists", "mood_quadrant", "llm_mood_label", "llm_mood_score"]]

🔍 **Discussion for students:** compare `mood_quadrant` (built from pure audio numbers) with `llm_mood_label` (built from *just the song title text*, with zero training!). They often disagree — which one do you trust more, and why? This is exactly the kind of feature debate that happens in real hackathons.

### Bonus: sentiment of the track title
A tiny, very fast sentiment model can also add a quick "title positivity" feature.


In [ ]:
# Load a tiny, very fast sentiment-analysis model (positive/negative)
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)

In [ ]:
# Grab just the track titles as a plain list of strings
titles = sample_df["track_name"].astype(str).tolist()

In [ ]:
# Run sentiment analysis on all titles at once (the pipeline handles batching for us)
sentiments = sentiment_pipe(titles)

In [ ]:
# Unpack the results into two new columns: the label (POSITIVE/NEGATIVE) and its confidence score
sample_df["title_sentiment_label"] = [s["label"] for s in sentiments]
sample_df["title_sentiment_score"] = [s["score"] for s in sentiments]

In [ ]:
# 👀 See how the model "feels" about each track title
sample_df[["track_name", "title_sentiment_label", "title_sentiment_score"]]

<a id="10"></a>
## 10. 🧩 Putting It All Together: The Final Feature Table

Let's see everything we engineered on the **full dataset** (excluding the HuggingFace embedding/zero-shot demo columns, which we only computed on samples for speed).


In [ ]:
# List every engineered column we want to keep in our final feature table
engineered_cols = [
    # identifiers
    "track_id", "track_name", "artists", "track_genre",
    # basic numeric
    "duration_min", "loudness_abs", "popularity_norm", "tempo_rounded",
    # ratio / interaction
    "energy_loudness_ratio", "dance_energy_score", "popularity_per_minute",
    "acoustic_instr_index", "vocal_energy_gap",
    # binning / encoding
    "duration_bucket", "tempo_bucket", "key_name", "mode_name",
    # text features
    "num_artists", "is_collab", "track_name_length", "track_word_count",
    "is_acoustic_version", "is_remix", "is_live", "has_parenthesis",
    # aggregation
    "genre_avg_popularity", "dance_rank_in_genre", "popularity_z_in_genre",
    "artist_avg_popularity",
    # music theory
    "mood_quadrant",
]

In [ ]:
# Build the final feature dataframe by selecting just those columns
df_features = df[engineered_cols].copy()

In [ ]:
# 👀 Check the shape and a preview of the final feature table
print("Final engineered feature table shape:", df_features.shape)
df_features.head(10)

In [ ]:
# 👀 Full statistical summary of every engineered feature
df_features.describe(include="all").T

<a id="14"></a>
## 14. 💾 Save the Engineered Dataset

We save the fully engineered table so it's ready to feed straight into **Week 4 (Supervised Learning)**.


In [ ]:
# Name the output file
OUTPUT_PATH = "spotify_kku_features_week2.csv"

In [ ]:
# Save the final engineered feature table to CSV
df_features.to_csv(OUTPUT_PATH, index=False)

In [ ]:
# 👀 Confirm it saved correctly
print(f"✅ Saved engineered dataset to: {OUTPUT_PATH}")
print("Shape:", df_features.shape)

<a id="15"></a>
## 15. ✅ Recap & What's Next

### What we did this week
- 🔁 Refreshed core Python tools: `def`, `lambda`, `if/elif/else`, `.map()`, `.apply()`, `.assign()`, list comprehensions
- 🔢 Built **numeric**, **ratio/interaction**, **binned/categorical**, **text-based**, **aggregation/group**, and **domain-knowledge (music theory)** features — all with pure pandas
- 🤗 Used **free HuggingFace models** to add **LLM-era features**: zero-shot mood classification, title sentiment, and semantic sentence embeddings
- 📉 Compressed high-dimensional embeddings into usable features with **PCA**
- 💾 Exported a clean, model-ready feature table

### Why this matters
> In real hackathons and Kaggle competitions, the teams that win are rarely the ones with the fanciest model — they're the ones with the **best features**. You now have a toolkit that spans classic pandas tricks all the way to modern LLM-powered feature extraction.

### 🔜 Coming up in Week 3: Supervised Learning
We'll take `spotify_kku_features_week2.csv` and:
- Split into train/test sets
- Train classification/regression models (e.g., predicting `track_genre` or `popularity`)
- Evaluate feature importance — and see which of today's features actually mattered!

---
**Author's note for instructors republishing this notebook:** feel free to fork, adapt, and reuse for your own class — please keep a reference back to the [CP020003_ArtificialIntelligence_2026s1](https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1) repository. Happy teaching! 🎓


---

<div style="
background: linear-gradient(135deg, #fafafa 0%, #eef6f9 50%, #e8eaf6 100%);
padding: 30px;
border-radius: 18px;
text-align: center;
font-family: 'Segoe UI', sans-serif;
box-shadow: 0 6px 18px rgba(0,0,0,0.06);
border: 1px solid #dce3ea;
">

  <h2 style="
  color: #5c6b8a;
  margin: 0 0 12px 0;
  font-size: 1.8em;
  font-weight: 700;">
  🎉 Well Done!
  </h2>

  <p style="
  color: #495057;
  font-size: 1.05em;
  margin: 6px 0;">
  You've completed the Week 2 Notebook for
  <strong style="color:#6c7aa1;">
  CP020003 — AI 2026 @ KKU
  </strong>
  </p>

  <!--
  <p style="
  color: #6c757d;
  font-size: 0.95em;
  margin-top: 12px;">
  Next week we dive into
  <strong style="color:#5b8def;">
  Supervised Learning
  </strong>
  — scikit-learn, train/test splits, and your first ML model 🚀
  </p>
  -->

  <hr style="
  border: 1px solid #c9d6df;
  width: 50%;
  margin: 16px auto;">

  <p style="
  color: #7d8790;
  font-size: 0.9em;
  font-style: italic;
  margin-bottom: 6px;">
  "Shared freely so that everyone, everywhere, can learn AI."
  </p>

  <p style="
  color: #8a97a6;
  font-size: 0.85em;">
  — Teerapong Panboonyuen (P'Kao) · teerapong.pa@chula.ac.th
  </p>

</div>

<a id="16"></a>
## 16. 📝 This Week's Assignment — 10 Feature Engineering Challenges

Now it's your turn! Below are **10 questions**, ordered from **easy → hard**, asking you to invent your own features using `pandas` on the same `df` we built in this notebook.

> 🧑‍🏫 **Instructor note:** each question below already contains a working solution under the `# TODO: Insert your code below` comment, so you can preview/test everything works before class. To turn this into a blank student assignment, simply delete the code under each `# --- SOLUTION` marker and leave the `# TODO` comment for students to fill in.

**How to submit:** run every cell top-to-bottom, make sure `df` ends up with all 10 new columns, then save your notebook and submit the `.ipynb` file.


### Question 1 (⭐ Easy) — Short Song Flag
Create a new boolean/int column `is_short_song` that equals `1` if a song's `duration_min` is **less than 2.5 minutes**, and `0` otherwise.

💡 *Hint: you already have `duration_min` from Section 3. Use a comparison operator.*


In [ ]:
# TODO: Insert your code below


### Question 2 (⭐ Easy) — Energy vs Valence Gap
Create a new column `energy_valence_diff` = `energy` minus `valence`. A high positive value means an "intense but not-so-happy" song (e.g. angry/aggressive), while a high negative value means "chill but happy".

💡 *Hint: simple column subtraction, just like `vocal_energy_gap` in Section 4.*


In [ ]:
# TODO: Insert your code below


### Question 3 (⭐⭐ Easy-Medium) — "Loud & Fast" Flag
Create a boolean/int column `loud_and_fast` that equals `1` only when **both**:
- `loudness_abs` is above the **median** loudness_abs of the whole dataset, **and**
- `tempo` is above the **median** tempo of the whole dataset

💡 *Hint: compute both medians first with `.median()`, then combine two conditions with `&`.*


In [ ]:
# TODO: Insert your code below


### Question 4 (⭐⭐ Medium) — Popularity Quartile Bucket
Using `pd.qcut`, split the `popularity` column into **4 equal-sized groups (quartiles)** and store the result in a new column `popularity_quartile`, labeled `["low", "mid-low", "mid-high", "high"]`.

💡 *Hint: `pd.qcut` splits by rank/percentile, unlike `pd.cut` which splits by fixed value ranges (see Section 5).*


In [ ]:
# TODO: Insert your code below

### Question 5 (⭐⭐ Medium) — Title Contains a Number
Create a boolean/int column `title_has_number` that equals `1` if the `track_name` contains **any digit** (0-9), e.g. "7 Rings" or "Track 10".

💡 *Hint: use `.str.contains()` with a regex digit pattern `r"\d"`, just like `is_acoustic_version` in Section 6.*


In [ ]:
# TODO: Insert your code below


### Question 6 (⭐⭐⭐ Medium) — Artist Track Count
Create a new column `artist_track_count` showing **how many tracks in this whole dataset** are credited to the exact same `artists` string (e.g. if "Jason Mraz" appears on 5 rows, every one of those rows should show `5`).

💡 *Hint: `.value_counts()` on the `artists` column gives you a lookup table — then use `.map()` to broadcast it back onto every row.*


In [ ]:
# TODO: Insert your code below


### Question 7 (⭐⭐⭐ Medium-Hard) — Genre Rank by Average Popularity
Create a new column `genre_rank_by_avg_popularity` where the **most popular genre overall** (by average `popularity`) gets rank `1`, the second most popular genre gets rank `2`, and so on — the **same rank number should repeat** for every row belonging to that genre.

💡 *Hint: this is different from `dance_rank_in_genre` in Section 7 — there we ranked **songs inside a genre**; here we rank **genres against each other**. Try: `groupby` + `.mean()` -> `.rank()` -> `.map()`.*


In [ ]:
# TODO: Insert your code below


### Question 8 (⭐⭐⭐⭐ Hard) — Duration Outlier Detection (per genre, IQR method)
For **each genre separately**, compute the IQR (Interquartile Range) of `duration_min` and flag a song as an outlier (`is_duration_outlier = 1`) if its duration falls outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` **for its own genre**.

💡 *Hint: this is the classic statistics "boxplot outlier rule", but computed per-group. Use `.groupby(...).transform(lambda x: x.quantile(...))` to get Q1 and Q3 broadcast back onto every row.*


In [ ]:
# TODO: Insert your code below


### Question 9 (⭐⭐⭐⭐ Hard) — Combined Mood-Tempo Category + Frequency Encoding
Create **two** new columns:
1. `mood_tempo_combo` — a string combining `mood_quadrant` and `tempo_bucket`, e.g. `"Happy_fast"` or `"Sad_slow"` (from Sections 8 and 5)
2. `mood_tempo_combo_count` — how many songs in the whole dataset share that exact same combo

💡 *Hint: string concatenation with `+` (remember to `.astype(str)` first!), then the same `.value_counts()` + `.map()` trick from Question 6.*


In [ ]:
# TODO: Insert your code below


### Question 10 (⭐⭐⭐⭐⭐ Hardest / Bonus) — Build a Music Mood Score

Imagine you are working as a Data Scientist at **Spotify**.

Your task is to design your own feature called **`music_mood_score`** that estimates **how happy and energetic a song feels**.

Your feature **must combine at least four existing numerical columns** from the dataset.

Possible columns include (but are not limited to):

- `danceability`
- `energy`
- `valence`
- `tempo`
- `acousticness`
- `speechiness`
- `liveness`
- `instrumentalness`

There is **no single correct formula**.

Instead, think like a real Data Scientist and design a feature that you believe best represents a song's overall mood.

### Requirements

1. Create a new column called `music_mood_score`.
2. Your formula must use **at least four numerical features**.
3. Display the **Top 10 songs** with the highest `music_mood_score`.
4. Briefly explain **why** you designed your formula this way.

💡 **Hint**

There is no perfect answer.

Your creativity and reasoning are more important than the exact formula.

In [ ]:
# TODO: Insert your code below


---
### 🎯 Assignment wrap-up
Congratulations — if you completed all 10, you've now practiced:
- Simple boolean/numeric features (Q1-Q2)
- Multi-condition logic (Q3)
- Quantile-based binning (Q4)
- Regex-based text features (Q5)
- Frequency/count encoding (Q6, Q9)
- Cross-group ranking (Q7)
- Statistical outlier detection per group (Q8)
- Categorical feature combination (Q9)
- Unsupervised learning as a feature source (Q10)

This is genuinely the **same toolkit real hackathon winners use** — nice work! See you next week for Supervised Learning. 🚀
